<a href="https://colab.research.google.com/github/NKVRK/Resume-Analyzer-using-FastAPI/blob/main/notebooks/en/code_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
%pip install -q faiss-cpu fastembed inflection tqdm

In [16]:
import os, json, re
from pathlib import Path
!unzip -o /content/Resume-Analyzer-using-FastAPI-main.zip -d /content

BASE_PATH = "/content/Resume-Analyzer-using-FastAPI-main"

BACKEND_ROOT = Path(BASE_PATH) / "backend"
FRONTEND_ROOT = Path(BASE_PATH) / "frontend"

EXCLUDE_FOLDERS = {"venv", ".venv", "node_modules", "__pycache__", ".git", "embeddings", "dist", "build", ".next"}

BACKEND_EXTS = {".py"}
FRONTEND_EXTS = {".js", ".jsx", ".ts", ".tsx"}

EMB_DIR = os.path.join(BASE_PATH, "embeddings")
os.makedirs(EMB_DIR, exist_ok=True)

TEXT_INDEX_PATH = os.path.join(EMB_DIR, "text_index.faiss")
TEXT_META_PATH  = os.path.join(EMB_DIR, "text_meta.jsonl")

BACKEND_INDEX_PATH = os.path.join(EMB_DIR, "backend_code_index.faiss")
BACKEND_META_PATH  = os.path.join(EMB_DIR, "backend_code_meta.jsonl")

FRONTEND_INDEX_PATH = os.path.join(EMB_DIR, "frontend_code_index.faiss")
FRONTEND_META_PATH  = os.path.join(EMB_DIR, "frontend_code_meta.jsonl")

print("Base Repo:", BASE_PATH)
print("Embeddings Location:", EMB_DIR)


Archive:  /content/Resume-Analyzer-using-FastAPI-main.zip
ad1eb4e07169145ed00caea86ecc9123039b2511
   creating: /content/Resume-Analyzer-using-FastAPI-main/
  inflating: /content/Resume-Analyzer-using-FastAPI-main/.gitignore  
  inflating: /content/Resume-Analyzer-using-FastAPI-main/README.md  
   creating: /content/Resume-Analyzer-using-FastAPI-main/backend/
   creating: /content/Resume-Analyzer-using-FastAPI-main/backend/app/
   creating: /content/Resume-Analyzer-using-FastAPI-main/backend/app/__pycache__/
  inflating: /content/Resume-Analyzer-using-FastAPI-main/backend/app/__pycache__/main.cpython-313.pyc  
   creating: /content/Resume-Analyzer-using-FastAPI-main/backend/app/api/
   creating: /content/Resume-Analyzer-using-FastAPI-main/backend/app/api/endpoints/
   creating: /content/Resume-Analyzer-using-FastAPI-main/backend/app/api/endpoints/__pycache__/
  inflating: /content/Resume-Analyzer-using-FastAPI-main/backend/app/api/endpoints/__pycache__/resumes.cpython-313.pyc  
  infla

In [17]:
def is_excluded(p: Path):
    return any(part in EXCLUDE_FOLDERS for part in p.parts)

def list_files(root, exts):
    if not root.exists(): return []
    files = []
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in exts and not is_excluded(p):
            files.append(str(p))
    return files

backend_files = list_files(BACKEND_ROOT, BACKEND_EXTS)
frontend_files = list_files(FRONTEND_ROOT, FRONTEND_EXTS)

print(f"Backend files: {len(backend_files)}")
print(f"Frontend files: {len(frontend_files)}")


Backend files: 8
Frontend files: 12


In [18]:
from tqdm import tqdm

def read_file(path):
    try: return Path(path).read_text(encoding="utf-8", errors="ignore")
    except: return ""

def chunk_lines(text, max_lines=40):
    lines = text.splitlines()
    return ["\n".join(lines[i:i+max_lines]) for i in range(0, len(lines), max_lines) if "".join(lines[i:i+max_lines]).strip()]

def to_structure(file_path, snippet, line_from, line_to):
    rel = os.path.relpath(file_path, BASE_PATH)
    module = Path(rel).parent.as_posix().replace("/", "_") or "root"

    return {
        "signature": snippet.splitlines()[0][:200] if snippet else "",
        "context": {
            "module": module,
            "file_path": rel,
            "file_name": Path(file_path).name,
            "snippet": snippet,
        },
        "line_from": line_from,
        "line_to": line_to,
    }

def build_structures(files):
    structs = []
    for fp in tqdm(files, desc="Chunking files"):
        txt = read_file(fp)
        start = 1
        for c in chunk_lines(txt, 40):
            line_count = c.count("\n") + 1
            structs.append(to_structure(fp, c, start, start + line_count - 1))
            start += line_count
    return structs

backend_structures = build_structures(backend_files)
frontend_structures = build_structures(frontend_files)

print("Backend chunks:", len(backend_structures))
print("Frontend chunks:", len(frontend_structures))


Chunking files: 100%|██████████| 12/12 [00:00<00:00, 3884.81it/s]

Backend chunks: 16
Frontend chunks: 23


In [19]:
import inflection

def textify(s):
    sig = inflection.humanize(inflection.underscore(s["signature"])) if s["signature"] else ""
    return f"Code section defined in module {s['context']['module']} with signature {sig}".strip()

text_structures = backend_structures + frontend_structures
text_representations = [textify(s) for s in text_structures]
print("Total text entries:", len(text_representations))


Total text entries: 39


In [20]:
from fastembed import TextEmbedding

BATCH_SIZE = 8

nlp_model  = TextEmbedding("sentence-transformers/all-MiniLM-L6-v2")
code_model = TextEmbedding("jinaai/jina-embeddings-v2-base-code")

print("✅ Models loaded")


✅ Models loaded


In [21]:
import faiss
import numpy as np

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r)+"\n")

def read_jsonl(path):
    return [json.loads(l) for l in open(path, "r", encoding="utf-8")]

def new_index(dim):
    return faiss.IndexFlatIP(dim)


In [22]:
def ensure_text_index():
    if os.path.exists(TEXT_INDEX_PATH) and os.path.exists(TEXT_META_PATH):
        print("✅ Loaded TEXT index")
        return faiss.read_index(TEXT_INDEX_PATH), read_jsonl(TEXT_META_PATH)

    print("Building TEXT index...")
    idx = None
    meta = []
    buf = []

    gen = nlp_model.embed(text_representations, batch_size=BATCH_SIZE)
    for emb, s in tqdm(zip(gen, text_structures), total=len(text_structures)):
        buf.append(emb)
        meta.append(s)
        if len(buf) >= BATCH_SIZE:
            vecs = np.array(buf, dtype="float32")
            faiss.normalize_L2(vecs)
            if idx is None: idx = new_index(vecs.shape[1])
            idx.add(vecs); buf=[]
    if buf:
        vecs = np.array(buf, dtype="float32")
        faiss.normalize_L2(vecs)
        if idx is None: idx = new_index(vecs.shape[1])
        idx.add(vecs)

    faiss.write_index(idx, TEXT_INDEX_PATH)
    write_jsonl(TEXT_META_PATH, meta)
    print("✅ TEXT index built")
    return idx, meta

text_index, text_meta = ensure_text_index()


Building TEXT index...


100%|██████████| 39/39 [00:00<00:00, 69.89it/s]

✅ TEXT index built


In [23]:
def ensure_backend_index():
    if os.path.exists(BACKEND_INDEX_PATH) and os.path.exists(BACKEND_META_PATH):
        print("✅ Loaded BACKEND code index")
        return faiss.read_index(BACKEND_INDEX_PATH), read_jsonl(BACKEND_META_PATH)

    print("Building BACKEND code index...")
    idx, meta, buf = None, [], []
    gen = code_model.embed([s["context"]["snippet"] for s in backend_structures], batch_size=BATCH_SIZE)

    for emb, s in tqdm(zip(gen, backend_structures), total=len(backend_structures)):
        buf.append(emb); meta.append(s)
        if len(buf) >= BATCH_SIZE:
            vecs = np.array(buf, dtype="float32")
            faiss.normalize_L2(vecs)
            if idx is None: idx = new_index(vecs.shape[1])
            idx.add(vecs); buf=[]
    if buf:
        vecs = np.array(buf, dtype="float32")
        faiss.normalize_L2(vecs)
        if idx is None: idx = new_index(vecs.shape[1])
        idx.add(vecs)

    faiss.write_index(idx, BACKEND_INDEX_PATH)
    write_jsonl(BACKEND_META_PATH, meta)
    print("✅ BACKEND code index built")
    return idx, meta

backend_index, backend_meta = ensure_backend_index()


Building BACKEND code index...


100%|██████████| 16/16 [00:31<00:00,  1.99s/it]

✅ BACKEND code index built


In [24]:
def ensure_frontend_index():
    if os.path.exists(FRONTEND_INDEX_PATH) and os.path.exists(FRONTEND_META_PATH):
        print("✅ Loaded FRONTEND code index")
        return faiss.read_index(FRONTEND_INDEX_PATH), read_jsonl(FRONTEND_META_PATH)

    print("Building FRONTEND code index...")
    idx, meta, buf = None, [], []
    gen = code_model.embed([s["context"]["snippet"] for s in frontend_structures], batch_size=BATCH_SIZE)

    for emb, s in tqdm(zip(gen, frontend_structures), total=len(frontend_structures)):
        buf.append(emb); meta.append(s)
        if len(buf) >= BATCH_SIZE:
            vecs = np.array(buf, dtype="float32")
            faiss.normalize_L2(vecs)
            if idx is None: idx = new_index(vecs.shape[1])
            idx.add(vecs); buf=[]
    if buf:
        vecs = np.array(buf, dtype="float32")
        faiss.normalize_L2(vecs)
        if idx is None: idx = new_index(vecs.shape[1])
        idx.add(vecs)

    faiss.write_index(idx, FRONTEND_INDEX_PATH)
    write_jsonl(FRONTEND_META_PATH, meta)
    print("✅ FRONTEND code index built")
    return idx, meta

frontend_index, frontend_meta = ensure_frontend_index()


Building FRONTEND code index...


100%|██████████| 23/23 [00:38<00:00,  1.67s/it]

✅ FRONTEND code index built


In [25]:
def emb_text(q):
    v = next(nlp_model.query_embed(q))
    v = np.array([v], dtype="float32")
    faiss.normalize_L2(v)
    return v

def emb_code(q):
    v = next(code_model.query_embed(q))
    v = np.array([v], dtype="float32")
    faiss.normalize_L2(v)
    return v

def run_search(index, meta, qvec, k=5):
    D, I = index.search(qvec, k)
    results = []
    for score, idx in zip(D[0], I[0]):
        if idx < 0: continue
        s = meta[idx]
        results.append((score, s["context"]["file_path"], s["context"]["snippet"]))
    return results

def show(results):
    for i,(score,fp,snip) in enumerate(results,1):
        print(f"\n#{i} score={score:.4f} :: {fp}\n{'-'*60}\n{snip[:800]}")


In [26]:
def search_backend_code(query):
    show(run_search(backend_index, backend_meta, emb_code(query)))

def search_frontend_code(query):
    show(run_search(frontend_index, frontend_meta, emb_code(query)))

def search_text(query):
    show(run_search(text_index, text_meta, emb_text(query)))


In [27]:
!ls -R ~/Downloads/Resume-Analyzer-using-FastAPI


/root/Downloads/Resume-Analyzer-using-FastAPI:
embeddings

/root/Downloads/Resume-Analyzer-using-FastAPI/embeddings:
backend_code_index.faiss  frontend_code_index.faiss  text_index.faiss
backend_code_meta.jsonl   frontend_code_meta.jsonl   text_meta.jsonl


In [31]:
search_backend_code("Where does the file's content get turned into plain text?")



#1 score=0.3018 :: frontend/src/index.js
------------------------------------------------------------
import React from 'react';
import ReactDOM from 'react-dom/client';
import './index.css';
import App from './App';
import reportWebVitals from './reportWebVitals';

const root = ReactDOM.createRoot(document.getElementById('root'));
root.render(
  <React.StrictMode>
    <App />
  </React.StrictMode>
);

// If you want to start measuring performance in your app, pass a function
// to log results (for example: reportWebVitals(console.log))
// or send to an analytics endpoint. Learn more: https://bit.ly/CRA-vitals
reportWebVitals();

#2 score=0.1863 :: frontend/src/App.test.js
------------------------------------------------------------
import { render, screen } from '@testing-library/react';
import App from './App';

test('renders learn react link', () => {
  render(<App />);
  const linkElement = screen.getByText(/learn react/i);
  expect(linkElement).toBeInTheDocument();
});

#3 score=